In [9]:
import os, re, time
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag
import requests

BASE_URL   = "https://patliputrabuilder.com/"
SAVE_DIR   = "downloaded_images"
MAX_PAGES  = 100     # stop after crawling this many pages
DELAY      = 0.3      # pause between requests, in seconds

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}
session = requests.Session()
session.headers.update(headers)

DOMAIN   = urlparse(BASE_URL).netloc
IMG_EXT  = re.compile(r"\.(jpe?g|png|gif|webp|svg|avif|bmp|ico)(\?.*)?$", re.I)
SKIP_EXT = re.compile(r"\.(pdf|zip|rar|mp4|mp3|docx?|xlsx?|pptx?|jpe?g|png|gif|webp|svg)(\?.*)?$", re.I)
BG_URL   = re.compile(r"url\(\s*['\"]?([^'\")]+)['\"]?\s*\)", re.I)

In [10]:
def extract_images(soup, page_url):
    """Find image URLs in <img>/<source> tags, srcset, lazy-load attributes,
    inline background images, <style> blocks and og:image meta tags."""
    found = set()
    add = lambda u: found.add(urljoin(page_url, u.strip())) if u and not u.strip().startswith("data:") else None

    for tag in soup.find_all(["img", "source"]):
        for attr in ("src", "data-src", "data-lazy-src", "data-original"):
            add(tag.get(attr))
        for attr in ("srcset", "data-srcset"):
            for part in (tag.get(attr) or "").split(","):
                add(part.strip().split(" ")[0])

    for tag in soup.find_all(style=True):
        for u in BG_URL.findall(tag["style"]):
            add(u)

    for style in soup.find_all("style"):
        for u in BG_URL.findall(style.get_text()):
            if IMG_EXT.search(u):
                add(u)

    for m in soup.find_all("meta", attrs={"property": ["og:image", "twitter:image"]}):
        add(m.get("content"))

    return found


In [12]:
from bs4 import BeautifulSoup
def normalize(url):
    url, _ = urldefrag(url)
    return url.rstrip("/")

visited, queue, all_images = set(), deque([BASE_URL]), set()

while queue and len(visited) < MAX_PAGES:
    url = normalize(queue.popleft())
    if url in visited:
        continue
    visited.add(url)

    try:
        r = session.get(url, timeout=15)
        if "text/html" not in r.headers.get("Content-Type", ""):
            continue
    except requests.RequestException as e:
        print("Skipped:", url, e)
        continue

    page = BeautifulSoup(r.text, "lxml")
    imgs = extract_images(page, url)
    all_images |= imgs
    print(f"[{len(visited)}] {url}  ->  {len(imgs)} images")

    for a in page.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("mailto:", "tel:", "javascript:", "+")):
            continue
        link = normalize(urljoin(url, href))
        p = urlparse(link)
        if p.scheme in ("http", "https") and p.netloc == DOMAIN \
                and link not in visited and not SKIP_EXT.search(p.path):
            queue.append(link)

    time.sleep(DELAY)

print(f"\nCrawled {len(visited)} pages, found {len(all_images)} unique images")


[1] https://patliputrabuilder.com  ->  43 images
[2] https://patliputrabuilder.com/about-us  ->  9 images
[3] https://patliputrabuilder.com/post-property  ->  10 images
[4] https://patliputrabuilder.com/career  ->  7 images
[5] https://patliputrabuilder.com/blogs  ->  6 images
[6] https://patliputrabuilder.com/contact-us  ->  6 images
[7] https://patliputrabuilder.com/Property/patliputra-signature-park  ->  6 images
[8] https://patliputrabuilder.com/blog/after-our-remarkable-success-in-bihar  ->  5 images
[9] https://patliputrabuilder.com/blog/now-in-greater-noida  ->  5 images
[10] https://patliputrabuilder.com/blog/offer-patliputra-signature-park  ->  5 images

Crawled 10 pages, found 54 unique images


In [13]:
os.makedirs(SAVE_DIR, exist_ok=True)

CT_EXT = {"image/jpeg": ".jpg", "image/png": ".png", "image/gif": ".gif",
          "image/webp": ".webp", "image/svg+xml": ".svg", "image/avif": ".avif"}

def safe_name(url, content_type=""):
    path = urlparse(url).path
    name = re.sub(r"[^\w.\-]", "_", os.path.basename(path)) or "image"
    if not os.path.splitext(name)[1]:
        name += CT_EXT.get(content_type.split(";")[0], ".jpg")
    # keep the folder name too, so files with the same name don't overwrite each other
    folder = os.path.basename(os.path.dirname(path))
    return f"{folder}_{name}" if folder else name

ok, skipped, failed = 0, 0, []
for i, img_url in enumerate(sorted(all_images), 1):
    try:
        r = session.get(img_url, timeout=20, stream=True)
        r.raise_for_status()
        ctype = r.headers.get("Content-Type", "")
        if not ctype.startswith("image/"):
            skipped += 1
            continue
        path = os.path.join(SAVE_DIR, safe_name(img_url, ctype))
        if os.path.exists(path):
            skipped += 1
            continue
        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        ok += 1
        print(f"[{i}/{len(all_images)}] saved  {path}")
    except Exception as e:
        failed.append((img_url, str(e)))
        print(f"[{i}/{len(all_images)}] FAILED {img_url}: {e}")
    time.sleep(0.1)

print(f"\nDone: {ok} downloaded, {skipped} skipped, {len(failed)} failed -> ./{SAVE_DIR}/")

[1/54] FAILED https://dev.patliputrabuilder.com/front/images/video-iframe.mp4: 404 Client Error: Not Found for url: https://dev.patliputrabuilder.com/front/images/video-iframe.mp4
[2/54] saved  downloaded_images\photo-1526289034009-0240ddb68ce3.jpg
[5/54] saved  downloaded_images\background_bg4.jpg
[6/54] saved  downloaded_images\images_bg-img.png
[7/54] saved  downloaded_images\delivered_alina.webp
[8/54] saved  downloaded_images\delivered_exotica.webp
[9/54] saved  downloaded_images\delivered_jyotipuram.webp
[10/54] saved  downloaded_images\delivered_lalita.webp
[11/54] saved  downloaded_images\delivered_madhuri.webp
[12/54] saved  downloaded_images\delivered_maharaja.webp
[13/54] saved  downloaded_images\delivered_mims.webp
[14/54] saved  downloaded_images\delivered_mussoorie.webp
[15/54] saved  downloaded_images\delivered_news1.webp
[16/54] saved  downloaded_images\delivered_news2.webp
[17/54] saved  downloaded_images\delivered_news4.webp
[18/54] saved  downloaded_images\delivered_

AttributeError: 'int' object has no attribute 'get_text'